# Open-Meteo data ingestion

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Data notebook 1 of 3 · ingestion*

Downloads hourly weather observations for **Sydney** from the Open-Meteo **Historical Weather API** and stores them unchanged in `data/raw/`.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Source

| Item | Value |
|:--|:--|
| Endpoint | `https://archive-api.open-meteo.com/v1/archive` (the Historical Weather API named in the brief) |
| Location | Sydney, NSW · latitude −33.8688, longitude 151.2093 |
| Period | 2016-01-01 to 2025-12-31 · **2026 onwards is reserved as production data and never downloaded** |
| Output | `data/raw/sydney_hourly.csv` |

### Variables requested

Only the variables named in the two target formulas are requested. Every other predictor used
later (lags, rolling windows, calendar encoding) is derived from these seven series.

| Variable | Unit | Climate Comfort Index | Weather Hazard Category |
|:--|:--|:--|:--|
| `temperature_2m` | °C | `TempScore` | `TempHazard` |
| `relative_humidity_2m` | % | `HumidityScore` | – |
| `wind_speed_10m` | km/h | `WindScore` | – |
| `cloud_cover` | % | `CloudScore` | `CloudHazard` |
| `precipitation` | mm | `RainScore` | `RainHazard` |
| `wind_gusts_10m` | km/h | – | `WindHazard` |
| `snowfall` | **cm** | – | `SnowHazard` |

> **Units matter.** They feed the formulas directly: `SnowHazard = MIN(snowfall / 15, 1)` is
> evaluated in **centimetres**, and the wind constants assume km/h. Units are logged from the API
> response at fetch time so they are verified, not assumed.

---
## 0. Setup

Set `REFRESH = True` for the first run (or after changing the variables or dates), then `False` to
reuse the saved file.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

PATHS = wx.paths()
RAW_CSV = PATHS["raw"] / "sydney_hourly.csv"

START_DATE, END_DATE = wx.DEFAULT_START, wx.DEFAULT_END   # 2026+ is reserved for production
REFRESH = True

print(f"Location  : {wx.SYDNEY.name} ({wx.SYDNEY.latitude}, {wx.SYDNEY.longitude})")
print(f"Window    : {START_DATE} to {END_DATE}")
print(f"Variables : {', '.join(wx.HOURLY_VARIABLES)}")
print(f"Output    : {RAW_CSV.relative_to(PATHS['root'])}")

---
## 1. Retrieval

`wx.fetch_hourly` wraps the API call with retries and three checks the API does not make itself:

| Check | Why it is needed |
|:--|:--|
| **Every variable present** | A missing variable is silently omitted from the response, and would otherwise surface much later as a `KeyError` |
| **Unambiguous timestamps** | Data is requested in **UTC** and converted to Sydney time. In local time the 2am hour repeats every April, so two different instants would share one timestamp; in UTC every timestamp is unique |
| **Exact hourly spacing** | Detects duplicated or skipped hours |

The saved CSV stores each timestamp with its UTC offset (e.g. `+11:00`), so it can be read back
without ambiguity using the generic `wx.load_file` and `wx.to_local_time`.

In [ ]:
if REFRESH:
    hourly = wx.fetch_hourly(start_date=START_DATE, end_date=END_DATE)
    wx.save_file(hourly, RAW_CSV, index=False)
    print(f"Downloaded and saved {len(hourly):,} rows.")
else:
    hourly = wx.load_file(RAW_CSV)
    hourly["time"] = wx.to_local_time(hourly["time"])
    print(f"Loaded {len(hourly):,} saved rows.")

hourly.head()

---
## 2. Coverage audit

The row count alone does not prove the series is usable: Open-Meteo pads unavailable periods with
nulls instead of shortening the response, and the archive lags real time by about five days.
This table establishes the usable window and per-column completeness — quote it in the report.

In [ ]:
coverage = wx.coverage_report(hourly)
print(f"Usable window: {coverage.attrs['first_observation']:%Y-%m-%d %H:%M} to "
      f"{coverage.attrs['last_observation']:%Y-%m-%d %H:%M}")
coverage.round(4)

---
## 3. Target sanity checks

Two properties of the formulas are checked before any modelling, because both shape the target
distributions:

- **Snowfall.** Sydney records no snow at sea level, so `SnowHazard` should always be zero. Its
  0.10 weight in the Weather Hazard Index is then unusable, which compresses the WHI range and
  pushes the classes toward Low Risk.
- **Hazard caps.** Each hazard term is capped at 1. The share of hours reaching the cap shows how
  much of the raw signal each formula discards.
- **Daylight saving.** Sydney's local calendar has 23-hour and 25-hour days; these are expected and
  are *complete* days.

In [ ]:
print(f"Snowfall: max {hourly['snowfall'].max()} cm, "
      f"non-zero hours {int((hourly['snowfall'] > 0).sum())}")

caps = pd.Series({
    "precipitation > 30 mm": (hourly["precipitation"] > 30).mean(),
    "wind_gusts_10m > 100 km/h": (hourly["wind_gusts_10m"] > 100).mean(),
    "snowfall > 15 cm": (hourly["snowfall"] > 15).mean(),
    "|temperature_2m - 22| > 25 °C": ((hourly["temperature_2m"] - 22).abs() > 25).mean(),
}, name="share")
display((caps * 100).round(4).to_frame("% of hours at the cap"))

hours_per_day = hourly.groupby(hourly["time"].dt.normalize()).size()
print("Local days by number of hours (23 and 25 are daylight-saving days):")
hours_per_day.value_counts().sort_index().rename_axis("hours_in_day").to_frame("days")

---
## 4. Findings and next steps

**Record in the report:**

- The usable window and any column below 100% completeness (section 2).
- That snowfall is always zero for Sydney, and what that means for the WHI weights and the class
  balance (section 3).
- How often each capped hazard term saturates, and the implication for the number of High and
  Extreme Risk days available for training.
- That the only non-24-hour days are daylight-saving transitions.

**Next:** the EDA notebook describes the data; the preprocessing notebook aggregates it to daily
values, builds both targets and the feature matrix, and writes the train/validation/test splits.